# Quantum-RAG Rating Predictor (QRRP): MovieLens (template)

Il construit un contexte RAG depuis l'historique utilisateur (voisins par similarité TF‑IDF sur *title|genres*) puis applique un raffinement quantique (Grover-like) sur une distribution 5‑classes, avant fusion.

> MovieLens n'a pas de texte review: l'entrée au LLM est *(film cible + métadonnées + historique condensé)*.


## 1) Imports (optionnel: installations)


In [ ]:
!pip -q install datasets pennylane scikit-learn transformers accelerate bitsandbytes

In [ ]:
# !pip -q install datasets pennylane scikit-learn transformers accelerate bitsandbytes

import os, math, json, random, re
import numpy as np
import pandas as pd

from typing import Dict, Tuple
from sklearn.metrics import mean_absolute_error, mean_squared_error


## 2) Configuration


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

MOVIELENS_VARIANT = "ml-100k"   # ex: "ml-1m"
N_EVAL = 300
K_NEIGHBORS = 8

# Quantum
N_QUBITS = 3
SHOTS = 400

# LLM (optionnel)
LLM_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
DEVICE = "cuda"


## 3) Charger MovieLens


In [ ]:
import os, zipfile, urllib.request
import pandas as pd

ML_URL = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
DATA_DIR = "./data_movielens"
ZIP_PATH = os.path.join(DATA_DIR, "ml-latest-small.zip")
EXTRACT_DIR = os.path.join(DATA_DIR, "ml-latest-small")

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(EXTRACT_DIR):
    if not os.path.exists(ZIP_PATH):
        print("Downloading MovieLens...")
        urllib.request.urlretrieve(ML_URL, ZIP_PATH)
    print("Extracting MovieLens...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(DATA_DIR)

ratings_path = os.path.join(EXTRACT_DIR, "ratings.csv")
movies_path  = os.path.join(EXTRACT_DIR, "movies.csv")

ratings = pd.read_csv(ratings_path)   # userId,movieId,rating,timestamp
movies  = pd.read_csv(movies_path)    # movieId,title,genres

df = ratings.merge(movies, on="movieId", how="left")

# Normalise types
df["userId"] = df["userId"].astype(int)
df["movieId"] = df["movieId"].astype(int)
df["rating"] = df["rating"].astype(float)

print(df.head())
print(df.shape)


In [ ]:
import numpy as np
import pandas as pd

def userwise_train_test_split(df, test_ratio=0.2, seed=42):
    rng = np.random.default_rng(seed)
    train_parts, test_parts = [], []
    for uid, g in df.groupby("userId"):
        idx = g.index.to_numpy()
        if len(idx) < 5:
            train_parts.append(g)
            continue
        n_test = max(1, int(len(idx) * test_ratio))
        test_idx = rng.choice(idx, size=n_test, replace=False)
        test_parts.append(df.loc[test_idx])
        train_parts.append(df.drop(index=test_idx).loc[idx[~np.isin(idx, test_idx)]])
    train = pd.concat(train_parts).reset_index(drop=True)
    test  = pd.concat(test_parts).reset_index(drop=True)
    return train, test

train_df, test_df = userwise_train_test_split(df, test_ratio=0.2, seed=42)
print(train_df.shape, test_df.shape)


In [ ]:
from collections import defaultdict

global_mean = train_df["rating"].mean()
user_mean = train_df.groupby("userId")["rating"].mean().to_dict()
item_mean = train_df.groupby("movieId")["rating"].mean().to_dict()

def predict_baseline(u, i, alpha=0.5):
    # mix user/item mean, fallback global
    um = user_mean.get(u, None)
    im = item_mean.get(i, None)
    if um is None and im is None:
        return global_mean
    if um is None:
        return im
    if im is None:
        return um
    return alpha*um + (1-alpha)*im

test_pred = test_df.apply(lambda r: predict_baseline(r["userId"], r["movieId"], alpha=0.5), axis=1).astype(float)
test_true = test_df["rating"].astype(float).to_numpy()


### Vérifier les colonnes requises (sinon merge movies)


In [ ]:
ratings_df = df  # alias explicite

required = {"userId","movieId","rating","title","genres"}
missing = required - set(ratings_df.columns)

if missing:
    raise ValueError(
        f"Colonnes manquantes {missing}. "
        "Si vous utilisez un MovieLens local, chargez movies.dat et faites un merge."
    )

print("OK – toutes les colonnes sont présentes")


## 4) RAG: voisins depuis l'historique utilisateur


In [ ]:
print("ratings_df columns:", ratings_df.columns.tolist())

sample_user = int(ratings_df["userId"].iloc[0])
sample_movie = int(ratings_df["movieId"].iloc[0])

neigh = retrieve_neighbors(sample_user, sample_movie, ratings_df, K_NEIGHBORS)
print("neigh columns:", neigh.columns.tolist())
neigh.head()


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

movie_meta = ratings_df[["movieId","title","genres"]].drop_duplicates("movieId").copy()
movie_meta["text"] = (movie_meta["title"].fillna("") + " | " + movie_meta["genres"].fillna("")).astype(str)

vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1,2))
X_items = vectorizer.fit_transform(movie_meta["text"].tolist())
movieid_to_index = {mid:i for i, mid in enumerate(movie_meta["movieId"].tolist())}

def retrieve_neighbors(user_id: int, target_movie_id: int, ratings: pd.DataFrame, k: int) -> pd.DataFrame:
    hist = ratings[(ratings["userId"]==user_id) & (ratings["movieId"]!=target_movie_id)]
    if hist.empty or target_movie_id not in movieid_to_index:
        return hist.head(0)

    t_idx = movieid_to_index[target_movie_id]
    hist_movie_ids = hist["movieId"].tolist()
    hist_idx = [movieid_to_index[m] for m in hist_movie_ids if m in movieid_to_index]
    if len(hist_idx)==0:
        return hist.head(0)

    sims = cosine_similarity(X_items[t_idx], X_items[hist_idx]).flatten()
    order = np.argsort(-sims)[:k]

    picked = hist.iloc[order].copy()
    picked["sim"] = sims[order]

    # IMPORTANT: no merge here, title/genres already present in ratings_df
    keep_cols = ["userId","movieId","rating","timestamp","title","genres","sim"]
    existing = [c for c in keep_cols if c in picked.columns]
    return picked[existing].sort_values("sim", ascending=False)


def format_rag_context(neigh: pd.DataFrame) -> str:
    if neigh is None or len(neigh)==0:
        return "No user history available."
    lines = []
    for _, row in neigh.iterrows():
        lines.append(
            "- {t} ({g}): user_rating={r}, sim={s:.3f}".format(
                t=row["title"], g=row["genres"], r=row["rating"], s=row["sim"]
            )
        )
    return "\n".join(lines)

sample_user = int(ratings_df["userId"].iloc[0])
sample_movie = int(ratings_df["movieId"].iloc[0])
neigh = retrieve_neighbors(sample_user, sample_movie, ratings_df, K_NEIGHBORS)
print(format_rag_context(neigh)[:800])


## 5) Quantum: Grover-like refinement sur 5 classes (optionnel)


In [ ]:
import pennylane as qml

def amplitude_encode_state(p5: np.ndarray) -> np.ndarray:
    amps = np.zeros(8, dtype=float)
    amps[:5] = np.sqrt(np.maximum(p5, 0.0))
    norm = np.linalg.norm(amps)
    if norm == 0:
        amps[0] = 1.0
        norm = 1.0
    return amps / norm

def grover_refine_distribution(p5: np.ndarray, target_i: int, shots: int) -> np.ndarray:
    dev = qml.device("default.qubit", wires=N_QUBITS, shots=shots)
    amps = amplitude_encode_state(p5)

    @qml.qnode(dev)
    def circuit():
        qml.StatePrep(amps, wires=range(N_QUBITS))
        qml.FlipSign(target_i, wires=range(N_QUBITS))   # oracle: un seul état marqué
        qml.GroverOperator(wires=range(N_QUBITS))       # diffusion
        return qml.probs(wires=range(N_QUBITS))

    probs8 = circuit()
    probs5 = np.array(probs8[:5], dtype=float)
    return probs5 / probs5.sum() if probs5.sum() else np.ones(5)/5


## 6) LLM (optionnel): chargement local


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

def load_llm(model_name: str, device: str):
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto" if device=="cuda" else None,
        torch_dtype="auto",
    )
    gen = pipeline(
        "text-generation",
        model=model,
        tokenizer=tok,
        max_new_tokens=128,
        do_sample=False,
        temperature=0.0,
    )
    return gen

gen = None
# gen = load_llm(LLM_MODEL_NAME, DEVICE)


## 7) Prompt RAG + prédiction


In [ ]:
SYSTEM_PROMPT = (
    "You are a rating prediction assistant. "
    "Return ONLY a valid JSON with keys rating (int 1..5), confidence (float 0..1), explanation (string)."
)

def build_prompt(user_id: int, target_movie_id: int, neigh: pd.DataFrame) -> str:
    row = movie_meta[movie_meta["movieId"]==target_movie_id].iloc[0]
    target_desc = f"{row['title']} | {row['genres']}"
    ctx = format_rag_context(neigh)
    parts = [
        "Task: Predict the user's rating (1..5) for the TARGET movie, given the user's past ratings.",
        f"USER_ID: {user_id}",
        f"TARGET_MOVIE: {target_desc}",
        "USER_HISTORY_NEIGHBORS (most similar items from user history):",
        ctx,
        "Guidelines: Use past ratings as preference signal; if history empty, best guess from genres.",
        "Output JSON only."
    ]
    return "\n".join(parts)

def parse_json(text: str) -> Dict:
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m:
        return {"rating": 3, "confidence": 0.0, "explanation": "parse_failed"}
    try:
        return json.loads(m.group(0))
    except Exception:
        return {"rating": 3, "confidence": 0.0, "explanation": "json_decode_failed"}

def llm_predict(prompt: str) -> Tuple[int, float, Dict]:
    if gen is None:
        return 3, 0.0, {"rating": 3, "confidence": 0.0, "explanation": "LLM not loaded; heuristic fallback"}
    out = gen(SYSTEM_PROMPT + "\n\n" + prompt)[0]["generated_text"]
    obj = parse_json(out)
    r = int(obj.get("rating", 3))
    r = min(max(r,1),5)
    c = float(obj.get("confidence", 0.0))
    c = float(np.clip(c, 0.0, 1.0))
    print("LLM raw output:", out[:400])
    print("Parsed obj:", obj)
    return r, c, obj

prompt = build_prompt(sample_user, sample_movie, neigh)
print(prompt[:700], "...\n")
print(llm_predict(prompt))


## 8) Prédiction hybride (LLM + quantum + fusion)


In [ ]:
def fusion(p_llm: np.ndarray, p_q: np.ndarray, lam: float) -> np.ndarray:
    lam = float(np.clip(lam, 0.0, 1.0))
    p = lam * p_llm + (1.0 - lam) * p_q
    return p / p.sum() if p.sum() else np.ones_like(p)/len(p)

def predict_one(user_id: int, movie_id: int, ratings_train: pd.DataFrame) -> Dict:
    neigh = retrieve_neighbors(user_id, movie_id, ratings_train, K_NEIGHBORS)
    prompt = build_prompt(user_id, movie_id, neigh)
    r_llm, c_llm, raw = llm_predict(prompt)

    # p^{LLM}: soft distribution centrée sur r_llm (tau dépend de confidence)
    tau = 1.0 + 2.0 * c_llm
    grid = np.arange(1,6)
    p_llm = np.exp(-tau * np.abs(grid - r_llm))
    p_llm = p_llm / p_llm.sum()

    # Quantum refinement + fusion
    p_q = grover_refine_distribution(p_llm, target_i=r_llm-1, shots=SHOTS)
    p_f = fusion(p_llm, p_q, lam=c_llm)
    r_hat = int(np.argmax(p_f) + 1)

    return {"userId": user_id, "movieId": movie_id, "rating_llm": r_llm, "confidence": c_llm,
            "rating_hat": r_hat, "y_true": None}

print(predict_one(sample_user, sample_movie, ratings_df))


## 9) Évaluation (holdout 1 item par utilisateur)


In [ ]:
def user_holdout_split(ratings: pd.DataFrame, seed: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(seed)
    tests = []
    trains = []
    for uid, grp in ratings.groupby("userId"):
        if len(grp) < 5:
            trains.append(grp)
            continue
        j = int(rng.integers(0, len(grp)))
        tests.append(grp.iloc[[j]])
        trains.append(grp.drop(grp.index[j]))
    return pd.concat(trains, ignore_index=True), pd.concat(tests, ignore_index=True)

train_df, test_df = user_holdout_split(ratings_df, SEED)
test_eval = test_df.sample(min(N_EVAL, len(test_df)), random_state=SEED).reset_index(drop=True)
print("train:", train_df.shape, "test_eval:", test_eval.shape)

rows = []
for i, row in test_eval.iterrows():
    out = predict_one(int(row["userId"]), int(row["movieId"]), train_df)
    out["y_true"] = float(row["rating"])
    rows.append(out)
    if (i+1) % 25 == 0:
        print(f"Done {i+1}/{len(test_eval)}")

res_df = pd.DataFrame(rows)
display(res_df.head())


## 10) Métriques + figures (style Amazon)


In [ ]:
import matplotlib.pyplot as plt

def metrics(y_true, y_pred) -> Dict[str,float]:
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    return {
        "RMSE": math.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "Accuracy": float(np.mean(np.round(y_true)==np.round(y_pred))),
    }

m_llm = metrics(res_df["y_true"], res_df["rating_llm"])
m_hyb = metrics(res_df["y_true"], res_df["rating_hat"])
print("LLM-only:", m_llm)
print("Hybrid:", m_hyb)

labels = ["Accuracy","MAE","RMSE"]
vals_llm = [m_llm["Accuracy"], m_llm["MAE"], m_llm["RMSE"]]
vals_hyb = [m_hyb["Accuracy"], m_hyb["MAE"], m_hyb["RMSE"]]

x = np.arange(len(labels))
w = 0.35

plt.figure(figsize=(7,4))
plt.bar(x-w/2, vals_llm, w, label="LLM-only")
plt.bar(x+w/2, vals_hyb, w, label="Hybrid (Q-LLM)")
plt.xticks(x, labels)
plt.legend()
plt.title("MovieLens: Overall metrics")
plt.tight_layout()
plt.show()

plt.figure(figsize=(5,5))
plt.scatter(res_df["y_true"], res_df["rating_hat"], alpha=0.6)
plt.xlabel("True rating")
plt.ylabel("Predicted rating")
plt.title("MovieLens: True vs Predicted (Hybrid)")
plt.tight_layout()
plt.show()


## 11) Export CSV


In [ ]:
out_path = "movielens_hybrid_results.csv"
res_df.to_csv(out_path, index=False)
print("Saved:", out_path)
